In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("ps07.ipynb")

# PS7 — Implementation Layers, Resources and Burden
### BioE 147/247 · Fall 2026

**Out:** Thursday, October 29 · **Due:** Thursday, November 5, 11:59 pm
**Covers:** Sessions 17, 18 and 19 · **48 points** · **Question 8** is required for **BioE 247** (56 total) and is extra credit for **BioE 147** (up to +8)

---

One thread: **the number you measured does not belong to the part.**

Session 17 said a part is manufactured, not found — the boundary around a
promoter is something you impose, and if you do not impose it the number
drifts. Session 18 said a parameter is manufactured too: an apparent Hill
coefficient of 26 out of one site, with the knob that places the switch
turning out to be a ratio of two expression levels. Session 19 said the thing
you cannot manufacture is a boundary around the ribosome pool — two genes with
no wire between them change each other, and there is no insulator to drop in.

Questions 1–2 are session 17, 3–5 are session 18, 6–8 are session 19.

Two of these pick up where an in-class sheet stopped: session 18's ledger — the trade between
sharpness and where the switch fires — is computed in Q3d,
and session 19's K-moves-with-demand diagnostic is Q6b.

**Collaboration is encouraged.** Discuss, argue, work at a whiteboard
together, then write your own solution and your own code. Record who you
worked with below.

**If you used an LLM**, say so briefly and say what for. The conditions are
that you can explain anything you submit and that the code you submit runs.

**A note on the visible tests.** They check *properties* — limits,
invariances, symmetries. A green visible check means "not obviously broken",
not "right".

Fill in the two cells below before you start.

In [ ]:
COLLABORATORS = ""   # e.g. "worked with J. Chen on Q6"
AI_USE = ""          # e.g. "used an LLM to check my algebra in Q3"

## Setup

In [ ]:
# ---------------------------------------------------------------------------
# SETUP — run this cell first, every time.
#
# DataHub / local : finds the repository root and puts it on the import path.
# Google Colab    : clones the repository first, because Colab opens this
#                   notebook on its own, without the posb package beside it.
# ---------------------------------------------------------------------------
import os
import sys

if "google.colab" in sys.modules:
    if not os.path.exists("posb2026"):
        !git clone -q https://github.com/ArkinLaboratory/posb2026.git
    sys.path.insert(0, os.path.abspath("posb2026"))
else:
    _d = os.getcwd()
    while _d != os.path.dirname(_d) and not os.path.isdir(os.path.join(_d, "posb")):
        _d = os.path.dirname(_d)
    sys.path.insert(0, _d)

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

import posb
from posb import Reaction, Model

posb.check_environment()


In [ ]:
from posb.assembly import revcomp, is_palindromic, junction_identities, distance
from posb.protein import (modified_fraction, crossing_fraction,
                          half_maximal_stimulus, stimulus_for_fraction,
                          sensitivity_index, apparent_hill)
from posb.resources import free_fraction, expression, isocost_sum

---
## Question 1 — The compositor, and where it breaks  *(6 points)*

Every abstraction this course has used — mass action, the quasi-steady-state
assumption, transfer curves, truth tables, and the compositor itself —
assumes that **composing A and B leaves A and B alone and combines their
behaviours**. Session 17 said that is false. This question makes you compute
how false, using the mechanism session 19 gave you.

A compositor is the code form of that assumption: hand it two characterised
parts and it returns the composite, by combining the numbers the parts
carry.

**Q1a.** Write `naive_compose(d1, d2)`, the compositor. Each gene has a
demand $d_i$, and a gene **on its own** reaches $p_i = d_i/(1+d_i)$. The
compositor assumes composition changes nothing, so it returns each gene's
alone-output, as a tuple `(p1, p2)`.

Then write `pool_compose(d1, d2)`, returning what the two genes **actually**
reach when they share one pool. Use `expression`.

In [ ]:
def naive_compose(d1, d2):
    """What a parts-list compositor predicts: each gene as if it were alone."""
    ...


def pool_compose(d1, d2):
    """What the two genes actually reach, sharing one pool."""
    ...


naive_compose(1.0, 4.0), pool_compose(1.0, 4.0)

In [ ]:
grader.check("q1a")

**Q1b.** Set `OVERSTATEMENT` to the factor by which the compositor
overstates **gene 1's** output at $d_1 = 1$, $d_2 = 4$ — its prediction
divided by what the gene actually reaches.

In [ ]:
...

OVERSTATEMENT

<!-- BEGIN QUESTION -->

**Q1c.** In writing:

**(i)** The compositor is not merely inaccurate here. An inaccurate model has
a true value it misses. Say what is wrong with this one in stronger terms
than that, in one sentence.

**(ii)** You could repair `naive_compose` by passing it the other gene's
demand. Say what that repair costs you as a design methodology — what can you
no longer do with a characterised part?

**(iii)** Session 17 listed the moves that make composition work anyway:
insulate, fix the junction, standardise the context. Name one that **cannot**
be applied to the failure you just computed, and say why.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

---
## Question 2 — Golden Gate, designed and repaired  *(7 points)*

A four-base overhang is a junction. Two overhangs that are reverse
complements of each other are **the same junction**, because the two strands
of one cut are one cut. You counted that space in class; here you search
it.

**Q2a.** You counted the space in class — 256 overhangs, 16 of them
palindromic and therefore unusable, 120 usable junction identities. Now use
it.

Write `compatible_with(chosen, min_distance=2)`, returning the **sorted list**
of overhangs you could still add to a reaction that already contains
`chosen`. An overhang qualifies when it is not palindromic, is not already in
`chosen`, and sits at distance at least `min_distance` from **every** member.

`posb.assembly.distance(a, b)` is the smaller of the distance from `a` to `b`
and the distance from `a` to the *reverse complement* of `b` — a fragment can
arrive either way round, so each overhang competes with every other
overhang's partner too. Using plain Hamming distance here will pass a set
that does not work at the bench, which is the usual way a hand-picked set
fails.

In [ ]:
from itertools import product

ALL_OVERHANGS = ["".join(s) for s in product("ACGT", repeat=4)]


def compatible_with(chosen, min_distance=2):
    """Overhangs that could still join a reaction already holding `chosen`."""
    ...


len(compatible_with(["GGAG", "AGGT", "GCTT"]))

In [ ]:
grader.check("q2a")

**Q2b.** Here is a proposed set for a four-fragment assembly. One of the
four is the problem. Set `MIN_DISTANCE` to the smallest pairwise distance in
the set, `SAFE` to whether it clears the distance-2 rule, and `REPAIRED` to a
four-overhang set that keeps the three good ones and replaces the bad one
with the **alphabetically first** overhang your `compatible_with` offers.

In [ ]:
PROPOSED = ["GGAG", "AGGT", "AATC", "GCTT"]

...

MIN_DISTANCE, SAFE, REPAIRED

<!-- BEGIN QUESTION -->

**Q2c.** Name the bad pair and say **which of the two terms** in `distance`
caught it — the direct comparison, or the reverse-complement one. Then say
what would go wrong at the bench: not "they clash", but what you would
actually see in the colonies.

**Q2d.** The published overhang sets were *reasoned* in 2015 from uniqueness
and non-palindromy — **not** from distance counting — and only *priced by
measurement* years later. When the measurement came, the distance rule turned
out to flag the bad pairs it found. So what is the rule good for, and what can
it still not tell you? *Two sentences.*

_Type your answer here, replacing this text._

<!-- END QUESTION -->

---
## Question 3 — The covalent cycle, and a ruler for sharpness  *(8 points)*

One kinase, one phosphatase, one conserved pool. With $w = W^*/W_T$ and
$k_i = K_{mi}/W_T$, the steady state satisfies

$$\frac{(V_1/V_2)(1-w)}{k_1 + 1 - w} = \frac{w}{k_2 + w}.$$

`posb.protein` carries this and nothing more: each function is one or two
lines of that algebra.

**Q3a.** The ruler itself you built in class: $R$ is the $V_1/V_2$ needed
to reach $w = 0.9$ divided by the $V_1/V_2$ needed to reach $w = 0.1$, a plain
hyperbola scores 81, and a Hill curve of coefficient $n$ scores $81^{1/n}$, so
$n = \ln 81 / \ln R$. `N_FROM_R` below is that line; it is given, because the
handout's answer sheet derives it.

What the *handout* could not do is let go of $k_1 = k_2$ — its sharpness item
fixes them equal. The session's own figure does not: it sweeps $k_2$ with
$k_1$ pinned, and the apparent Hill coefficient falls from about 51 to about 3
as the second enzyme is relaxed. **The property is in neither enzyme.** Here
you put numbers on that.

Take $k_1 = 0.02$, $k_2 = 0.002$ — one enzyme ten times more saturated than
the other — and set `R_ASYM` and `N_ASYM` to its sensitivity index and
apparent Hill coefficient. Then set `N_GEO` for the symmetric pair sitting at
their **geometric mean**, $k_1 = k_2 = \sqrt{0.02 \times 0.002} = 0.0063246$:
the same scale, with the asymmetry taken out.

In [ ]:
def N_FROM_R(R):
    """Apparent Hill coefficient of a dose-response scoring R on the ruler."""
    return np.log(81.0) / np.log(R)


...

R_ASYM, N_ASYM, N_GEO

In [ ]:
grader.check("q3a")

**Q3b.** Sweep $k_1 = k_2 = k$ over `K_SWEEP` and record the apparent Hill
coefficient at each. Put the answers in `N_SWEEP`, same order.

In [ ]:
K_SWEEP = np.array([1.0, 0.3, 0.1, 0.03, 0.01])

...

list(zip(K_SWEEP, np.round(N_SWEEP, 2)))

In [ ]:
grader.check("q3b")

**Q3c.** All four response curves in segment 2 of session 18 met at a
single point, at $V_1/V_2 = 1$, and that crossing sits at
$w = k_2/(k_1+k_2)$. Since $k_i = K_{mi}/W_T$, the pool size cancels out of it
exactly.

Set `CROSSING_LO` and `CROSSING_HI` to the crossing fraction at
`W_total = 0.1` and at `W_total = 1000`, holding $K_{m1} = 0.2$ and
$K_{m2} = 0.05$ fixed. The functions take $k_1$ and $k_2$, so you form those
yourself; any correct route is fine.

Then look at the value. Every curve you were shown in class had
$k_1 = k_2$, so they all crossed at one half. **This pair does not.** One half
is a property of the symmetric case, not of the cycle.

In [ ]:
K_M1, K_M2 = 0.2, 0.05

...

CROSSING_LO, CROSSING_HI

In [ ]:
grader.check("q3c")

**Q3d — the trade, computed.** The session showed you the whole reachable
band: for each target sharpness, every threshold you could reach with any
choice of affinities, and the band closing as the switch gets sharper. Here
you put two named designs inside it and read off the corners.

Hold the two enzymes unequal — $k_1 = 4k_2$ throughout, since at $k_1 = k_2$
the half-maximal stimulus is exactly 1 whatever $k$ is and there is nothing to
see — and compare:

| | $k_1$ | $k_2$ |
|---|---|---|
| the blunt one  | 0.163128 | 0.040782 |
| the sharp one  | 0.017060 | 0.004265 |

Set `N_BLUNT`, `U50_BLUNT`, `N_SHARP`, `U50_SHARP`.

In [ ]:
...

(N_BLUNT, U50_BLUNT), (N_SHARP, U50_SHARP)

In [ ]:
grader.check("q3d")

<!-- BEGIN QUESTION -->

**Q3e.** Two sentences, on the numbers you just computed.

**(i)** The sharp design is six times steeper and its firing point has moved
most of the way to 1. Say what a designer loses, concretely — what could you
do with the blunt cycle that you cannot do with the sharp one?

**(ii)** The cycle burns ATP continuously at rest, with the output not
moving, and it burns most where it is sharpest. Say why both of those are
true of the same mechanism.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

---
## Question 4 — CRISPRi is session 6 with a different occupant  *(5 points)*

In the weak-promoter limit, a promoter blocked by a dCas9–sgRNA complex at
concentration $C$ with dissociation constant $K_d$ has

$$F_{\text{reg}} = \frac{1}{1 + C/K_d}, \qquad\text{so}\qquad
\text{fold repression} = 1 + \frac{C}{K_d}.$$

That is the occupancy function from 15 September with a different thing
sitting on the DNA. Qi *et al.* publish no model; this one is ours.

There is no `posb.crispri`. Write it.

**Q4a.** Write both functions below.

In [ ]:
def fold_repression(C, K_d):
    """Fold repression of a promoter blocked by a complex at concentration C."""
    ...


def guide_fraction(demands):
    """Fraction of ONE shared dCas9 pool captured by each guide.

    `demands` is a sequence of C_i/K_di, one per guide. Returns an array that
    sums to the occupied fraction of the pool, not to 1.
    """
    ...


fold_repression(10.0, 1.0), guide_fraction([1.0, 1.0, 2.0])

In [ ]:
grader.check("q4a")

<!-- BEGIN QUESTION -->

**Q4b.** The specificity of a CRISPRi repressor comes from 20 nt of base
pairing, so $K_d$ is something you **design** rather than something you
inherit from a protein–DNA interface. Name one engineering consequence of
that which is not available to a protein repressor.

**Q4c.** $C$ — the concentration of the dCas9–sgRNA complex — is **not yours
to set independently per guide**. Say why, and say what happens to the fold
repression of an existing guide when you add a second one to the cell.

**Q4d.** You have now written the same denominator three times this
semester: operator occupancy in session 6, decoy titration in session 18, and
a shared dCas9 pool here. Say in one sentence what the three have in common
at the level of mechanism.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

---
## Question 5 — A memory element with no differential equation  *(2 points)*

Session 16 built an SR latch: the state is which of two gates is holding, you
write it with a pulse on S or R, and what holds it is the loop continuing to
drive itself.

A recombinase memory element stores the same bit as the **orientation of a
DNA segment**. You write it by expressing integrase; you write it back by
expressing integrase together with its directionality factor.

<!-- BEGIN QUESTION -->

**Q5.** In writing, no code. The side-by-side comparison was done in
class — do not reproduce it. These are the three things it raises and does
not settle:

**(a)** What holds the state is the difference that matters: a latch is held
by a loop that keeps driving itself, a recombinase by the sequence of the
DNA. Say what that buys you, and name the cost it imposes that the latch does
not have.

**(b)** The recombinase model is a state table with a rate per cell per hour,
and there is no differential equation anywhere in it. Explain why an ODE
would be the wrong model here — not merely unnecessary, but wrong.

**(c)** Measured switching takes hours, against seconds to minutes for the
latch. Say which you would choose to record a rare event in a population you
will assay a week later, and why the slow one can be the right answer.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

---
## Question 6 — The edge that is not in the diagram  *(12 points)*

With $d_i = m_i/K_i$ the demand of gene $i$,

$$p_i \;\propto\; \frac{d_i}{1 + \sum_j d_j}.$$

**Q6a — the direction you will actually be in.** In class you went
forwards: demands in, fall out. At the bench you have the fall and you want
the demand.

Write `infer_demand(d1, ratio)`, returning the competitor's demand $d_2$ that
would drive gene 1 down to `ratio` times its uncompeted output, given gene
1's own demand $d_1$. Then set `D2_OBSERVED` for a gene at $d_1 = 1$ measured
at **40%** of what it reaches alone.

In [ ]:
def infer_demand(d1, ratio):
    """The competitor demand implied by an observed fall to `ratio`."""
    ...


...

D2_OBSERVED

In [ ]:
grader.check("q6a")

**Q6b — the diagnostic, and the competitor you forgot about.**

A fall like that is *exactly* the shape of repression by a non-cooperative
repressor, and nothing is repressing anything. One number separates them: the
half-point of the apparent repression curve moves when you change the
**reporter's** own expression, and a real dissociation constant cannot.

In the two-gene cell you worked in class the half-point is $1 + d_1$. Real
cells are not two-gene cells. Write `K_app(d1, d_other)` for a cell in which
your reporter runs at $d_1$, the competitor you are titrating is the one you
care about, and **a third construct you did not account for** sits there at
demand `d_other`, pulling the whole time.

Then run the diagnostic. You titrate against the same reporter at $d_1 = 0.5$
and $d_1 = 3$, in a strain that also carries a selection marker at
$d_{\text{other}} = 2$. Set `K_LOW`, `K_HIGH`, `K_RATIO`, and `K_RATIO_NAIVE`
— the ratio you would have predicted had you assumed the cell held only your
two genes.

In [ ]:
def K_app(d1, d_other=0.0):
    """The competitor demand at which gene 1's output halves."""
    ...


...

K_LOW, K_HIGH, K_RATIO, K_RATIO_NAIVE

In [ ]:
grader.check("q6b")

**Q6c — look inside the model the closed form came from.**

Everything above is the closed form. `posb.resources` also carries the
binding model it is the steady state *of*: `competition_model(demands, k=...)`
returns a `posb.core.Model` whose state is the **bound complexes** $C_i$ and
the outputs $p_i$. Free ribosomes are not a species — they are whatever
conservation leaves, $R_{\text{free}} = 1 - \sum_i C_i$.

Integrate it from an empty cell at $d_1 = 1$, $d_2 = 4$, with a translation
rate of 2 so the outputs are not numerically equal to the complexes, and read
off the **occupancy**: `SIM_C1`, `SIM_C2`, `SIM_FREE`, and `SIM_P1`.

    m = competition_model([1.0, 4.0], k=2.0)
    tr = m.simulate({sp: 0.0 for sp in m.species}, (0.0, 400.0))
    #   tr["C1"] is the trajectory of C1; tr["C1"][-1] is its last value

The closed form gives you $p$. It does not give you how much of the pool is
tied up, or in what proportion — and that is the quantity a designer is
actually spending.

In [ ]:
from posb.resources import competition_model

...

SIM_C1, SIM_C2, SIM_FREE, SIM_P1

In [ ]:
grader.check("q6c")

<!-- BEGIN QUESTION -->

**Q6d.** One sentence, no code, and read the model before you answer.

`competition_model` writes translation as $C_i \to C_i + p_i$: the ribosome is
**not released** when it makes a protein. So $\dot C_i$ is binding minus
unbinding and nothing else, and the binding equilibrium holds **exactly** at
steady state — at any binding rate, fast or slow. That is why the closed form
and the simulation agree to machine precision rather than approximately.

Say what the quasi-steady-state assumption is therefore an assumption *about*
— since it is plainly not needed for the fixed point — and name one thing
about this cell the model as written cannot represent.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

**Q6e.** You have the diagnostic as arithmetic. Now cost it as an
experiment: name what you would actually vary at the bench to change the
reporter's own demand **without cloning anything**, and name one artefact
that could move the apparent half-point for a reason that has nothing to do
with the ribosome pool.

**Q6f.** Gyorgy *et al.* found that inducing a gene on a plasmid dropped a
reporter on the **same plasmid** by up to 60% and left a **chromosomal**
reporter unmoved — and built a separate strain to show that a chromosomal
drop would have been detectable. Why is that control the difference between a
result and an absence of evidence, and what does the result say about where
the pool ends?

**Q6g.** Your diagnostic assumes one pool. Given Q6d, say what it would
report if the competitor and the reporter sat on different replicons, and
whether that is a false negative or a correct answer to a different
question.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

---
## Question 7 — The response time you already computed  *(8 points)*

Session 5: $t_{1/2} = \ln 2/(\gamma + \mu)$, and PS2 used $\mu$ as a number
the medium hands you.

**Q7a.** Set `T_HALF_FREE` to the response time of a protein with
$\gamma = 0.1$/h in a cell growing at $\mu = 0.6$/h, and `T_HALF_BURDENED` to
the same protein's response time once the circuit is on and $\mu$ has fallen
to 0.35/h. Hours.

In [ ]:
GAMMA = 0.1

...

T_HALF_FREE, T_HALF_BURDENED

In [ ]:
grader.check("q7a")

**Q7b.** Set `AFFECTED` to the string `"circuit"`, `"host"` or `"both"`,
naming **whose** proteins have their response time changed when your circuit
switches on — yours, the host's, or both.

This asks only whose. *Which* of them feel it most is Q7e, and it is a
different question with a different answer.

In [ ]:
...

AFFECTED

In [ ]:
grader.check("q7b")

**Q7c — burden does not slow a circuit down. It re-times it.**

Your circuit has two arms. One carries an ssrA tag and is degraded fast,
$\gamma_{\text{fast}} = 2.0$/h. The other is stable and goes only by dilution,
$\gamma_{\text{slow}} = 0.05$/h. Switching the circuit on takes $\mu$ from
0.6/h to 0.35/h.

Set `T_FAST_OFF`, `T_FAST_ON`, `T_SLOW_OFF`, `T_SLOW_ON` to the four response
times, and `DELAY_OFF`, `DELAY_ON` to the **gap between the two arms** —
slow minus fast — before and after.

In [ ]:
GAMMA_FAST, GAMMA_SLOW = 2.0, 0.05
MU_OFF, MU_ON = 0.60, 0.35

...

(T_FAST_OFF, T_FAST_ON), (T_SLOW_OFF, T_SLOW_ON), (DELAY_OFF, DELAY_ON)

In [ ]:
grader.check("q7c")

<!-- BEGIN QUESTION -->

**Q7d.** The gap between your two arms grew by a factor of about 1.8 when
the circuit switched on, and the circuit did the switching. Sessions 8 to 11
were spent on what a relative delay does to a two-arm circuit. Name one
circuit from those sessions whose behaviour depends on that gap, and say what
this does to it.

**Q7e.** The *direction* of all of this is unambiguous and the *magnitude* is
not. Name what you would have to measure, in your own strain and your own
medium, before you could put a number on the penalty — and say why a figure
taken from a paper will not do.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

---
## Question 8 — required for BioE 247, extra credit for BioE 147

**BioE 247** — part of the assignment, worth 8 of your 56
points.

**BioE 147** — optional, worth up to 8 points of extra credit on top of
48. You do not need it for full marks. It is not a harder version of the
same thing; it removes an assumption the earlier questions made, which is
where most of the interest is.

### The growth laws

Two straight lines. Change the **nutrient** and the ribosomal protein fraction
rises with growth rate. Hold the nutrient and **inhibit translation**, and it
rises as growth *falls*.

<!-- BEGIN QUESTION -->

**Q8.** In writing:

**(a)** A cell is slowed to half its growth rate two different ways: a poorer
carbon source, and chloramphenicol. Which way does $\varphi_R$ move in each
case, and why are they opposite?

**(b)** Schaechter, Maaløe and Kjeldgaard (1958) slowed cells to about half
speed a *third* way — they dropped the temperature from 37 °C to 25 °C — and
found that mass, RNA, DNA and nuclei per cell stayed **nearly constant for a
given medium**. Given (a), what does that rule out about what the cell is
sensing?

**(c)** Scott *et al.* model a useless protein as a neutral component and
predict $\lambda/\lambda_0 = 1 - \varphi_U/\varphi_c$ with
$\varphi_c \approx 0.48$. Estimate the growth cost of expressing a useless
protein at 10% of total protein, and say what the 0.48 intercept means
physically — it is not 1.0, and that is the interesting part.

**(d)** The model charges for proteome fraction and **nothing else** — not
amino acids, not ATP per peptide bond, not chaperone capacity. Chemically
unrelated proteins, made from different expression systems, fall on the same
relation. Explain why that is the evidence which licenses the simplification,
and name one regime where you would expect it to fail.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

---
## Before you submit

1. `Kernel → Restart Kernel and Run All Cells`
2. Confirm it runs top to bottom with no errors
3. Submit the `.ipynb` to Gradescope

A notebook that works only because of leftover state from cells run in a
different order is not a working notebook.

In [ ]:
grader.check_all()